# Extrema of a Random Spline
We synthesize a spline of a specified degree, delay, period, with random spline coefficients. We display this spline in <span style="color:#1f77b4">**blue**</span>, with blue stems and rings that highlight the samples at the integers, and red stems and rings that highlight the samples at the boundaries of one period. The black dots give the knots of the spline.

We then extract from this random spline the list of the intervals where the spline takes extremal values, which we overlay as thick lines and markers in the <span style="color:#3eb489">**mint**</span> color for the minima and in the <span style="color:#fdbe02">**mango**</span> color for the maxima. Finally, we print a verbose description of the minima and the maxima.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Persistent spline
f = sk.PeriodicSpline1D()
f.spline_coeff[0] = rng.standard_normal()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    show_minima = True,
    show_maxima = True
):
    global f # Spline

    # Update the period while maintaining the samples
    y = f.get_samples(f.delay, support_length = f.period)
    if f.period < period:
        y = np.append(y, rng.standard_normal(period - f.period))
    else:
        y = y[ : period]
    f = sk.PeriodicSpline1D.from_samples(y, degree = degree).delayed_by(delay)
    # Plot the spline
    (fig, ax) = plt.subplots()
    f.plot((fig, ax), plotpoints = 301)

    # Extremal pieces
    [minima, maxima] = f.extrema()
    # Plot each piece independently
    def show_extrema (
        extrema,
        color
    ):
        for x in extrema:
            lb = x.domain.infimum # Lower bound of the domain
            ub = x.domain.supremum # Upper bound of the domain
            if not x.domain.isleftopen:
                ax.plot(
                    lb,
                    x.value,
                    marker = "o",
                    markerfacecolor = color,
                    markeredgecolor = color,
                    markersize = 11.0
                )
            if not x.domain.isrightopen:
                ax.plot(
                    ub,
                    x.value,
                    marker = "o",
                    markerfacecolor = color,
                    markeredgecolor = color,
                    markersize = 11.0
                )
            if lb != ub:
                ax.plot([lb, ub], [x.value, x.value], color, linewidth = 9.0)
    if show_minima:
        show_extrema(minima, "#3eb489")
    if show_maxima:
        show_extrema(maxima, "#fdbe02")
    # Show the plot
    plt.show()

    # Verbose description
    print("---")
    if show_minima: # Minima
        print("# Minima")
        for x in minima:
            print(x)
    if show_maxima: # Maxima
        print("# Maxima")
        for x in maxima:
            print(x)

# Interactions
minima_checkbox_widget = widgets.Checkbox(
    value = True,
    description = "Show minima"
)
maxima_checkbox_widget = widgets.Checkbox(
    value = True,
    description = "Show maxima"
)
widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    show_minima = minima_checkbox_widget,
    show_maxima = maxima_checkbox_widget
)